In [2]:
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

data = pd.read_pickle('../data/merged_clean.pkl')
label_col = "Label"
y = data[label_col]
X = data.drop(columns=[label_col])

constant_cols = [c for c in X.columns if X[c].nunique() <= 1]
X = X.drop(columns=constant_cols + ['Destination Port', 'Fwd Header Length.1'])

y = y.replace({
    'Web Attack - Brute Force': 'Web Attack',
    'Web Attack - XSS': 'Web Attack',
    'Web Attack - Sql Injection': 'Web Attack',
})
mask = ~y.isin(['Heartbleed', 'Infiltration'])
X, y = X[mask], y[mask]

X = X.astype(np.float32)

print("Shape:", X.shape, " Classes:", y.nunique())
print(y.value_counts())

Shape: (2520751, 68)  Classes: 11
Label
BENIGN              2095057
DoS Hulk             172846
DDoS                 128014
PortScan              90694
DoS GoldenEye         10286
FTP-Patator            5931
DoS slowloris          5385
DoS Slowhttptest       5228
SSH-Patator            3219
Web Attack             2143
Bot                    1948
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

print("Train:", X_train.shape)
print("Test: ", X_test.shape)
print("\nTraining class counts:")
print(y_train.value_counts())

Train: (1764525, 68)
Test:  (756226, 68)

Training class counts:
Label
BENIGN              1466539
DoS Hulk             120992
DDoS                  89610
PortScan              63486
DoS GoldenEye          7200
FTP-Patator            4152
DoS slowloris          3769
DoS Slowhttptest       3660
SSH-Patator            2253
Web Attack             1500
Bot                    1364
Name: count, dtype: int64


In [5]:
from imblearn.over_sampling import SMOTE
from collections import Counter
import time

TARGET = 50_000

counts = Counter(y_train)
# Only oversample classes below the target; leave larger ones untouched
strategy = {cls: TARGET for cls, n in counts.items() if n < TARGET}

print("Classes to oversample:")
for cls, n in sorted(strategy.items()):
    print(f"  {cls:20s} {counts[cls]:>7,} -> {n:,}")

t = time.time()
sm = SMOTE(sampling_strategy=strategy, random_state=42, k_neighbors=5)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print(f"\nSMOTE completed in {time.time()-t:.1f}s")

print("\nAfter SMOTE:")
print(pd.Series(y_train_res).value_counts())
print("\nTraining rows:", len(y_train_res), "(was", len(y_train), ")")


Classes to oversample:
  Bot                    1,364 -> 50,000
  DoS GoldenEye          7,200 -> 50,000
  DoS Slowhttptest       3,660 -> 50,000
  DoS slowloris          3,769 -> 50,000
  FTP-Patator            4,152 -> 50,000
  SSH-Patator            2,253 -> 50,000
  Web Attack             1,500 -> 50,000

SMOTE completed in 30.0s

After SMOTE:
Label
BENIGN              1466539
DoS Hulk             120992
DDoS                  89610
PortScan              63486
Web Attack            50000
FTP-Patator           50000
DoS GoldenEye         50000
DoS Slowhttptest      50000
SSH-Patator           50000
DoS slowloris         50000
Bot                   50000
Name: count, dtype: int64

Training rows: 2090627 (was 1764525 )


In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, classification_report
import time

# Without SMOTE
t = time.time()
dt_plain = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print(f"Plain DT trained in {time.time()-t:.1f}s")
pred_plain = dt_plain.predict(X_test)

# With SMOTE
t = time.time()
dt_smote = DecisionTreeClassifier(random_state=42).fit(X_train_res, y_train_res)
print(f"SMOTE DT trained in {time.time()-t:.1f}s")
pred_smote = dt_smote.predict(X_test)

print(f"\nMacro F1 without SMOTE: {f1_score(y_test, pred_plain, average='macro'):.4f}")
print(f"Macro F1 with SMOTE:    {f1_score(y_test, pred_smote, average='macro'):.4f}")

print("\n=== WITHOUT SMOTE ===")
print(classification_report(y_test, pred_plain, digits=4))
print("\n=== WITH SMOTE ===")
print(classification_report(y_test, pred_smote, digits=4))

Plain DT trained in 343.0s
SMOTE DT trained in 338.1s

Macro F1 without SMOTE: 0.9725
Macro F1 with SMOTE:    0.9658

=== WITHOUT SMOTE ===
                  precision    recall  f1-score   support

          BENIGN     0.9995    0.9990    0.9993    628518
             Bot     0.8057    0.8168    0.8112       584
            DDoS     0.9994    0.9999    0.9997     38404
   DoS GoldenEye     0.9926    0.9935    0.9930      3086
        DoS Hulk     0.9986    0.9980    0.9983     51854
DoS Slowhttptest     0.9507    0.9585    0.9546      1568
   DoS slowloris     0.9932    0.9889    0.9910      1616
     FTP-Patator     0.9989    0.9983    0.9986      1779
        PortScan     0.9890    0.9995    0.9942     27208
     SSH-Patator     0.9887    0.9938    0.9912       966
      Web Attack     0.9629    0.9689    0.9659       643

        accuracy                         0.9987    756226
       macro avg     0.9708    0.9741    0.9725    756226
    weighted avg     0.9987    0.9987    0.998

In [7]:
from sklearn.metrics import classification_report
import pandas as pd

rep_plain = pd.DataFrame(classification_report(y_test, pred_plain, output_dict=True)).T
rep_smote = pd.DataFrame(classification_report(y_test, pred_smote, output_dict=True)).T
rep_plain.to_csv('../results/dt_no_smote_report.csv')
rep_smote.to_csv('../results/dt_smote_report.csv')
print("Saved both reports")

Saved both reports
